In [4]:
from tokenizers import Tokenizer, models, pre_tokenizers, normalizers, decoders, processors
from transformers import PreTrainedTokenizerFast

# https://huggingface.co/docs/tokenizers/python/latest/pipeline.html#
# https://machinelearningplus.com/gen-ai/train-custom-tokenizer-huggingface/

# import
bpe = models.BPE.from_file(
    "vocab.json",
    "merges.txt",
    unk_token="<unk>")

tok = Tokenizer(bpe)

# Normalizer must match what was done training time
tok.normalizer = normalizers.Sequence([
    normalizers.NFC(), #standard
    normalizers.Lowercase(), #since tok training lowercased
])

# pre-tokenization (must match training too -> am i missing some other splitting?)
tok.pre_tokenizer = pre_tokenizers.Whitespace()

# postprocessing: adding special tokens
tok.post_processor = processors.TemplateProcessing(
    single="<bos> $A <eos>",
    pair="<bos> $A <eos> $B:1 <eos>:1",
    special_tokens=[
        ("<bos>", tok.token_to_id("<bos>")),
        ("<eos>", tok.token_to_id("<eos>")),],
)

# wrap in HF tokenizer -> here we will add other special tokens
hf_tok = PreTrainedTokenizerFast(
    tokenizer_object=tok,
    unk_token="<unk>",
    pad_token="<pad>",
    bos_token="<bos>",
    eos_token="<eos>",
)

In [5]:
# save in folder 
hf_tok.save_pretrained("morph_aware_tok/")
print("Morph-aware tokenizer hf compatible to morph_aware_tok/")

Morph-aware tokenizer hf compatible to morph_aware_tok/


In [6]:
## inspect id matching
enc = hf_tok("La canción es buena")
print(enc.tokens())          # inspect the subword splits
print(enc["input_ids"])
print(hf_tok.decode(enc["input_ids"]))

['<bos>', 'la', 'c', 'an', 'ción', 'es', 'buen', 'a', '<eos>']
[2, 564, 153, 74, 184, 375, 150, 18, 3]
<bos> la c an ción es buen a <eos>
